In [ ]:
# Starter code

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()
print(words[:8])
len(words)

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


32033

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [4]:
block_size = 3 # context length

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     
Xdev, Ydev = build_dataset(words[n1:n2])   
Xte,  Yte  = build_dataset(words[n2:])     

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [6]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) 
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((block_size*n_embd, n_hidden), generator=g)
b1 = torch.randn(n_hidden, generator=g)
W2 = torch.randn((n_hidden, vocab_size), generator=g)
b2 = torch.randn(vocab_size, generator=g)
parameters = [C, W1, b1, W2, b2]

print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

11897


In [7]:
max_steps=200000
batch_size=32
lossi=[]
for i in range (max_steps):

    # minibatch construct
    ix = torch.randint(0,Xtr.shape[0], (batch_size,) ) 
    Xb, Yb = Xtr[ix], Ytr[ix] # batch X ,Y
    # forward pass
    emb = C[Xb] 
    embcat=emb.view(emb.shape[0], -1)
    hpreact= embcat @ W1 +b1
    h=torch.tanh(hpreact) 
    logits= h @ W2 + b2
    loss= F.cross_entropy(logits,Yb)
    
    # backward pass
    for p in parameters:
        p.grad=None
    loss.backward()

    #update
    lr=0.1 if i <100000 else 0.01
    for p in parameters:
        p.data+= -lr*p.grad 
    
    # track stats:
    if i % 1000==0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item(): 4f}')
   
    lossi.append(loss.log10().item())
print(loss.item())

      0/ 200000:  28.404825
   1000/ 200000:  4.211333
   2000/ 200000:  3.052796
   3000/ 200000:  2.582946
   4000/ 200000:  2.942914
   5000/ 200000:  2.631311
   6000/ 200000:  2.462884
   7000/ 200000:  2.795535
   8000/ 200000:  2.584225
   9000/ 200000:  3.032491
  10000/ 200000:  2.538113
  11000/ 200000:  2.580786
  12000/ 200000:  3.075134
  13000/ 200000:  2.372739
  14000/ 200000:  2.459730
  15000/ 200000:  2.512594
  16000/ 200000:  2.649664
  17000/ 200000:  2.252535
  18000/ 200000:  2.295392
  19000/ 200000:  2.554878
  20000/ 200000:  2.639405
  21000/ 200000:  2.761427
  22000/ 200000:  2.628031
  23000/ 200000:  2.560265
  24000/ 200000:  2.909552
  25000/ 200000:  2.799705
  26000/ 200000:  2.357139
  27000/ 200000:  2.676391
  28000/ 200000:  2.810483
  29000/ 200000:  2.665242
  30000/ 200000:  2.349008
  31000/ 200000:  2.072838
  32000/ 200000:  2.890427
  33000/ 200000:  2.907489
  34000/ 200000:  2.294879
  35000/ 200000:  2.396714
  36000/ 200000:  2.634388


In [9]:
@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  h = torch.tanh(embcat @ W1 +b1) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.1381280422210693
val 2.176238775253296


In [10]:
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

carmahxato.
harifi.
mili.
taty.
skaalsten.
zhetralee.
rha.
kaeli.
nermara.
chriiv.
kaleigh.
ham.
pori.
quint.
shon.
rai.
adbi.
wanelogiefrynix.
kael.
dura.
